# 01 — Data Acquisition

**Project:** Causal ML for Climate Policy Effectiveness  
**Purpose:** Programmatically download all required datasets and organize them into `data/raw/`  
**Output:** Populated `data/raw/` directory + `data/raw/data_manifest.json`

---

## Datasets to Acquire

| # | Dataset | Source | Format |
|---|---------|--------|--------|
| 1 | Our World in Data — CO₂ & GHG Emissions | GitHub (OWID) | CSV |
| 2 | Our World in Data — Energy | GitHub (OWID) | CSV |
| 3 | Climate TRACE — Sector Emissions | climatetrace.org | CSV |
| 4 | Climate Policy Database | climatepolicydatabase.org | CSV |
| 5 | World Bank Development Indicators | World Bank API | CSV |
| 6 | World Bank Governance Indicators (WGI) | World Bank | CSV |

## Environment Setup

In [1]:
import os
import sys
import json
import hashlib
import zipfile
import io
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from tqdm.notebook import tqdm

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# Create directories
for subdir in ["owid", "climate_trace", "climate_policy_db", "world_bank", "oecd"]:
    (RAW_DIR / subdir).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {RAW_DIR}")

Project root: F:\Nexus_Projects\causal-climate-policy
Data directory: F:\Nexus_Projects\causal-climate-policy\data\raw


In [2]:
# Watermark for reproducibility
%load_ext watermark
%watermark -v -p pandas,requests,numpy

Python implementation: CPython
Python version       : 3.10.11
IPython version      : 8.39.0

pandas  : 2.3.3
requests: 2.34.2
numpy   : 2.0.2



## Helper Functions

In [3]:
def compute_sha256(filepath: Path) -> str:
    """Compute SHA-256 hash of a file for data integrity verification."""
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)
    return sha256.hexdigest()


def download_file(url: str, dest_path: Path, force: bool = False, desc: str = None) -> Path:
    """
    Download a file from URL with progress bar and caching.
    Skips download if file already exists (unless force=True).
    """
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)

    if dest_path.exists() and not force:
        size_mb = dest_path.stat().st_size / (1024 * 1024)
        print(f"   Already exists ({size_mb:.1f} MB): {dest_path.name}")
        return dest_path

    print(f"    Downloading: {desc or url}")
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))
    with open(dest_path, "wb") as f:
        with tqdm(total=total_size, unit="B", unit_scale=True, desc=dest_path.name) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))

    size_mb = dest_path.stat().st_size / (1024 * 1024)
    print(f"   Saved ({size_mb:.1f} MB): {dest_path}")
    return dest_path


# Data manifest to track all downloads
manifest = {}

---
## 1.1 — Our World in Data: CO₂ & GHG Emissions

The OWID CO₂ dataset is the most comprehensive open dataset combining:
- Historical CO₂ emissions by country (from the Global Carbon Project)
- Emissions by source (coal, oil, gas, cement, flaring)
- Per capita, per GDP metrics
- Population and GDP data

**Source:** [github.com/owid/co2-data](https://github.com/owid/co2-data)

In [4]:
print("=" * 60)
print("1.1 — Our World in Data: CO₂ & GHG Emissions")
print("=" * 60)

# Main dataset
owid_co2_path = download_file(
    url="https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv",
    dest_path=RAW_DIR / "owid" / "owid-co2-data.csv",
    desc="OWID CO₂ Data (main)",
)

# Codebook
owid_codebook_path = download_file(
    url="https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-codebook.csv",
    dest_path=RAW_DIR / "owid" / "owid-co2-codebook.csv",
    desc="OWID CO₂ Codebook",
)

# Quick validation
df_co2 = pd.read_csv(owid_co2_path)
print(f"\n Shape: {df_co2.shape}")
print(f" Year range: {df_co2['year'].min()} – {df_co2['year'].max()}")
print(f" Countries/entities: {df_co2['country'].nunique()}")
print(f" Columns: {df_co2.shape[1]}")

manifest["owid_co2"] = {
    "path": str(owid_co2_path),
    "url": "https://github.com/owid/co2-data",
    "description": "Our World in Data — CO₂ and Greenhouse Gas Emissions",
    "rows": len(df_co2),
    "columns": df_co2.shape[1],
    "sha256": compute_sha256(owid_co2_path),
    "downloaded_at": datetime.now().isoformat(),
}

1.1 — Our World in Data: CO₂ & GHG Emissions
   Already exists (13.7 MB): owid-co2-data.csv
   Already exists (0.0 MB): owid-co2-codebook.csv



 Shape: (50411, 79)
 Year range: 1750 – 2024
 Countries/entities: 254
 Columns: 79


---
## 1.2 — Our World in Data: Energy

Energy mix data is critical as a confounder — a country's emission trajectory depends
heavily on its energy composition (fossil vs. renewable share).

**Source:** [github.com/owid/energy-data](https://github.com/owid/energy-data)

In [5]:
print("=" * 60)
print("1.2 — Our World in Data: Energy")
print("=" * 60)

owid_energy_path = download_file(
    url="https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv",
    dest_path=RAW_DIR / "owid" / "owid-energy-data.csv",
    desc="OWID Energy Data",
)

df_energy = pd.read_csv(owid_energy_path)
print(f"\n Shape: {df_energy.shape}")
print(f" Year range: {df_energy['year'].min()} – {df_energy['year'].max()}")
print(f" Countries/entities: {df_energy['country'].nunique()}")

manifest["owid_energy"] = {
    "path": str(owid_energy_path),
    "url": "https://github.com/owid/energy-data",
    "description": "Our World in Data — Energy dataset",
    "rows": len(df_energy),
    "columns": df_energy.shape[1],
    "sha256": compute_sha256(owid_energy_path),
    "downloaded_at": datetime.now().isoformat(),
}

1.2 — Our World in Data: Energy
   Already exists (8.8 MB): owid-energy-data.csv



 Shape: (23377, 130)


 Year range: 1900 – 2025
 Countries/entities: 314


---
## 1.3 — Climate TRACE: Sector Emissions

Climate TRACE provides the most granular, independent emissions dataset available —
with sector-level breakdowns (power, transport, industry, agriculture, etc.) by country.

**Source:** [climatetrace.org/downloads](https://climatetrace.org/downloads)

> **Note:** Climate TRACE data is packaged as ZIP files by country or sector. 
> We download the country-level aggregated data.

In [6]:
print("=" * 60)
print("1.3 — Climate TRACE: Sector Emissions")
print("=" * 60)

# Climate TRACE provides a country-level summary download
# The direct download URL for the country overview data
CLIMATE_TRACE_URL = "https://api.climatetrace.org/v7/rankings/countries?since=2015&to=2024"

ct_dest = RAW_DIR / "climate_trace" / "climate_trace_country_emissions.json"

if not ct_dest.exists():
    print("    Fetching Climate TRACE country emissions via API...")
    try:
        response = requests.get(CLIMATE_TRACE_URL, timeout=60)
        response.raise_for_status()
        with open(ct_dest, "w") as f:
            json.dump(response.json(), f, indent=2)
        print(f"   Saved: {ct_dest}")
    except requests.exceptions.RequestException as e:
        print(f"    API request failed: {e}")
        print("   Manual download instructions:")
        print("     1. Visit https://climatetrace.org/downloads")
        print("     2. Download 'Country emissions' CSV package")
        print(f"     3. Extract to: {RAW_DIR / 'climate_trace'}")
else:
    print(f"   Already exists: {ct_dest.name}")



1.3 — Climate TRACE: Sector Emissions
   Already exists: climate_trace_country_emissions.json


---
## 1.4 — Climate Policy Database

The Climate Policy Database (by NewClimate Institute) is the most comprehensive
open dataset of national climate policies, containing 5,000+ policies with:
- Policy type (carbon tax, renewable mandate, efficiency standard, etc.)
- Year of adoption
- Target sector
- Country

**Source:** [climatepolicydatabase.org](https://www.climatepolicydatabase.org/)

> **Note:** The Climate Policy Database requires a manual CSV export from their website.
> We provide instructions and a fallback using the IEA PAMS dataset.

In [7]:
print("=" * 60)
print("1.4 — Climate Policy Database")
print("=" * 60)

cpd_path = RAW_DIR / "climate_policy_db" / "climate_policy_database_policies_export.csv"

if not cpd_path.exists():
    print("   MANUAL DOWNLOAD REQUIRED:")
    print("     1. Visit: https://www.climatepolicydatabase.org/policies")
    print("     2. Use filters or select all policies")
    print("     3. Click 'Export' → CSV")
    print(f"     4. Save as: {cpd_path}")
else:
    df_policies = pd.read_csv(cpd_path, dtype=str)
    print(f"   Already exists: {cpd_path.name} ({len(df_policies)} rows)")

manifest["climate_policy_db"] = {
    "path": str(cpd_path),
    "url": "https://www.climatepolicydatabase.org/",
    "description": "Climate Policy Database — National climate policies",
    "rows": len(df_policies) if 'df_policies' in dir() else "unknown",
    "downloaded_at": datetime.now().isoformat(),
}

1.4 — Climate Policy Database


   Already exists: climate_policy_database_policies_export.csv (6727 rows)


---
## 1.5 — World Bank Development Indicators

Key socioeconomic confounders from the World Bank:
- GDP per capita (constant 2015 USD)
- Population
- Urbanization rate
- Trade openness (% of GDP)
- Industry value added (% of GDP)

**Source:** [World Bank API via `wbgapi`](https://data.worldbank.org/)

In [8]:
print("=" * 60)
print("1.5 — World Bank Development Indicators")
print("=" * 60)

import wbgapi as wb

# Define indicators to download
WB_INDICATORS = {
    "NY.GDP.PCAP.KD": "gdp_per_capita",          # GDP per capita (constant 2015 US$)
    "SP.POP.TOTL": "population",                   # Population, total
    "SP.URB.TOTL.IN.ZS": "urban_pct",             # Urban population (% of total)
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",            # Trade (% of GDP)
    "NV.IND.TOTL.ZS": "industry_pct_gdp",         # Industry (including construction), value added (% of GDP)
    "EG.USE.PCAP.KG.OE": "energy_use_per_capita",  # Energy use (kg of oil equivalent per capita)
    "EN.ATM.CO2E.PC": "co2_per_capita_wb",         # CO2 emissions (metric tons per capita)
}

wb_path = RAW_DIR / "world_bank" / "wb_indicators.csv"

if not wb_path.exists():
    print("    Fetching World Bank indicators via API...")
    try:
        # Fetch data for all countries, years 1990-2024
        df_wb = wb.data.DataFrame(
            list(WB_INDICATORS.keys()),
            time=range(1990, 2025),
            labels=False,
            columns="series",
            numericTimeKeys=True,
        )
        
        # Reset index to get country and year as columns
        df_wb = df_wb.reset_index()
        df_wb = df_wb.rename(columns={"economy": "iso3", "time": "year"})
        df_wb = df_wb.rename(columns=WB_INDICATORS)
        
        df_wb.to_csv(wb_path, index=False)
        print(f"   Saved: {wb_path.name}")
        print(f"   Shape: {df_wb.shape}")
        print(f"   Countries: {df_wb['iso3'].nunique()}")
        print(f"   Years: {df_wb['year'].min()} – {df_wb['year'].max()}")
    except Exception as e:
        print(f"    World Bank API failed: {e}")
        print("   Will use GDP/population from OWID dataset as fallback")
else:
    df_wb = pd.read_csv(wb_path)
    print(f"   Already exists: {wb_path.name} ({len(df_wb)} rows)")

manifest["world_bank"] = {
    "path": str(wb_path),
    "url": "https://data.worldbank.org/",
    "description": "World Bank Development Indicators — GDP, population, urbanization, trade",
    "indicators": list(WB_INDICATORS.values()),
    "downloaded_at": datetime.now().isoformat(),
}

1.5 — World Bank Development Indicators


   Already exists: wb_indicators.csv (9310 rows)


---
## 1.6 — Worldwide Governance Indicators (WGI)

Governance quality is a critical moderator — policies may be less effective
in countries with weak institutions. The WGI provides six governance dimensions:
- Regulatory Quality
- Rule of Law
- Government Effectiveness
- Control of Corruption
- Voice and Accountability
- Political Stability

**Source:** [info.worldbank.org/governance/wgi](https://info.worldbank.org/governance/wgi/)

In [9]:
print("=" * 60)
print("1.6 — Worldwide Governance Indicators (WGI)")
print("=" * 60)

WGI_INDICATORS = {
    "RQ.EST": "regulatory_quality",      # Regulatory Quality: Estimate
    "RL.EST": "rule_of_law",             # Rule of Law: Estimate
    "GE.EST": "govt_effectiveness",      # Government Effectiveness: Estimate
    "CC.EST": "control_corruption",      # Control of Corruption: Estimate
}

wgi_path = RAW_DIR / "world_bank" / "wgidataset_with_sourcedata-2025.xlsx"

if not wgi_path.exists():
    print("   MANUAL DOWNLOAD REQUIRED:")
    print("     1. Visit: https://info.worldbank.org/governance/wgi/")
    print("     2. Click 'Data Access -> Full Dataset'")
    print(f"     3. Save as: {wgi_path}")
else:
    print(f"   Already exists: {wgi_path.name}")

manifest["wgi_governance"] = {
    "path": str(wgi_path),
    "url": "https://info.worldbank.org/governance/wgi/",
    "description": "Worldwide Governance Indicators — Regulatory quality, rule of law, etc.",
    "indicators": list(WGI_INDICATORS.values()),
    "downloaded_at": datetime.now().isoformat(),
}

1.6 — Worldwide Governance Indicators (WGI)
   Already exists: wgidataset_with_sourcedata-2025.xlsx


---
## 1.7 — Data Manifest

Save a comprehensive manifest logging all data sources, download dates,
checksums, and row counts for full reproducibility.

In [10]:
print("=" * 60)
print("1.7 — Saving Data Manifest")
print("=" * 60)

manifest_path = RAW_DIR / "data_manifest.json"

manifest["_metadata"] = {
    "project": "Causal ML for Climate Policy Effectiveness",
    "created_at": datetime.now().isoformat(),
    "python_version": sys.version,
    "total_datasets": len([k for k in manifest if not k.startswith("_")]),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"   Manifest saved: {manifest_path}")
print(f"\n Summary of acquired datasets:")
print(f"{'─' * 50}")
for key, info in manifest.items():
    if key.startswith("_"):
        continue
    rows = info.get("rows", "N/A")
    desc = info.get("description", "")[:50]
    print(f"  {key:25s} | {str(rows):>8s} rows | {desc}")

1.7 — Saving Data Manifest
   Manifest saved: F:\Nexus_Projects\causal-climate-policy\data\raw\data_manifest.json

 Summary of acquired datasets:
──────────────────────────────────────────────────
  owid_co2                  |    50411 rows | Our World in Data — CO₂ and Greenhouse Gas Emissio
  owid_energy               |    23377 rows | Our World in Data — Energy dataset
  climate_policy_db         |     6727 rows | Climate Policy Database — National climate policie
  world_bank                |      N/A rows | World Bank Development Indicators — GDP, populatio
  wgi_governance            |      N/A rows | Worldwide Governance Indicators — Regulatory quali


---
## 1.8 — Validation Summary

Quick validation of all downloaded files.

In [11]:
print("=" * 60)
print("1.8 — Validation Summary")
print("=" * 60)

val_paths = {
    "OWID CO₂": RAW_DIR / "owid" / "owid-co2-data.csv",
    "OWID Energy": RAW_DIR / "owid" / "owid-energy-data.csv",
    "Climate Policies": RAW_DIR / "climate_policy_db" / "climate_policy_database_policies_export.csv",
    "World Bank WDI": RAW_DIR / "world_bank" / "wb_indicators.csv",
    "WGI Governance": RAW_DIR / "world_bank" / "wgidataset_with_sourcedata-2025.xlsx",
}

print("\nDataset              Status           Size       Rows")
print("─" * 55)

all_ok = True
for name, p in val_paths.items():
    if p.exists():
        size_mb = p.stat().st_size / (1024 * 1024)
        
        # Row counting for validation. NOTE: raw newline counting
        # (previously `sum(1 for _ in f) - 1`) overcounts whenever a CSV
        # field contains an embedded newline inside a quoted value (common
        # in the Climate Policy Database's free-text description fields),
        # which produced a mismatched row count vs. the pandas-parsed count
        # used elsewhere (e.g. Section 1.4). Use pandas' CSV parser so the
        # count here matches the row count actually loaded downstream.
        if p.suffix == ".csv":
            rows = len(pd.read_csv(p, dtype=str))
        elif p.suffix == ".xlsx":
            rows = "Excel"
        else:
            rows = "N/A"
            
        print(f"{name:20s} [OK]        {size_mb:>7.1f} MB  {str(rows):>9s}")
    else:
        print(f"{name:20s} [MISSING]      N/A        N/A")
        all_ok = False

print("\n" + "=" * 55)
if all_ok:
    print("  All core datasets successfully acquired! Ready for Notebook 02.")
else:
    print("  Some datasets are missing — check instructions above")


1.8 — Validation Summary

Dataset              Status           Size       Rows
───────────────────────────────────────────────────────


OWID CO₂             [OK]           13.7 MB      50411


OWID Energy          [OK]            8.8 MB      23377
Climate Policies     [OK]            5.1 MB       6727
World Bank WDI       [OK]            0.8 MB       9310
WGI Governance       [OK]            9.9 MB      Excel

  All core datasets successfully acquired! Ready for Notebook 02.
